# BIAN 310 · Module 3 — Usman Solutions Data Cleaning & Automation

**Freshly rebuilt Google Colab student notebook — 11 November 2026**

Complete the actual **Usman Solutions** case using the Knoxville and Boise files provided in the LMS.

**Workflow**  
1. Enter your full name and the assignment title.  
2. Upload `Case Problem 1, Usman Solutions.zip` (recommended) or both `knoxville.txt` and `boise.txt`.  
3. Combine the two markets and preserve original records.  
4. Flag missing values, invalid responses and duplicate candidates.  
5. Compute Taste only from complete Saltiness, Sweetness and Flavor scores.  
6. Filter respondents **under 35** who selected **No** for regular Crispy Spuds consumption.  
7. Generate the required `porter_hinkle.xlsx` and `young_no.xlsx`.  
8. Generate an inline visual dashboard showing survey findings and data-quality problems.
9. Export both Excel workbooks, audit files, PNG/HTML dashboard and summary to one ZIP; Colab triggers download automatically.

**Important:** This notebook helps perform the mechanics but does not replace your own review, decisions or 150–200-word management summary. Suspected duplicate rows are **never silently deleted**, and missing survey responses are **never fabricated**.

**Start here:** Choose **Runtime → Run all** or run each code cell with the ▶ button. When prompted, enter your details and upload the case ZIP. If Colab blocks an automatic download, open the left **Files** pane and download the ZIP from `/content/`.


## Step 1 — Student details (required)

In [ ]:
from pathlib import Path
import re

STUDENT_NAME = input("Enter your full name: ").strip()
while not STUDENT_NAME:
    STUDENT_NAME = input("Name is required. Enter your full name: ").strip()
ASSIGNMENT = input("Enter assignment name [Module 3 - Usman Solutions]: ").strip() or "Module 3 - Usman Solutions"
COURSE = "BIAN 310 - Business Intelligence & Analytics"
SAFE_NAME = re.sub(r"[^A-Za-z0-9_-]+", "_", STUDENT_NAME).strip("_") or "Student"
print(f"Welcome, {STUDENT_NAME}!\nCourse: {COURSE}\nAssignment: {ASSIGNMENT}")

## Step 2 — Upload the source ZIP or both `.txt` files

Upload one ZIP from **General Resources**, or upload both text files in the same chooser. The notebook looks for the filenames and detects tab delimiters. It does not require a Kaggle path.

In [ ]:
import io, zipfile
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

if IN_COLAB:
    print("Choose the original ZIP or select BOTH knoxville.txt and boise.txt.")
    UPLOADED = files.upload()  # {filename: bytes}
else:
    print("Outside Colab: set LOCAL_SOURCE_ZIP to the case ZIP, or LOCAL_SOURCE_DIR to the text files.")
    LOCAL_SOURCE_ZIP = Path("Case Problem 1, Usman Solutions.zip")
    LOCAL_SOURCE_DIR = Path(".")
    if LOCAL_SOURCE_ZIP.exists():
        UPLOADED = {LOCAL_SOURCE_ZIP.name: LOCAL_SOURCE_ZIP.read_bytes()}
    else:
        UPLOADED = {p.name: p.read_bytes() for p in LOCAL_SOURCE_DIR.glob("*.txt")}

source_bytes = {}
for filename, blob in UPLOADED.items():
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(blob)) as case_zip:
            for member in case_zip.namelist():
                base = Path(member).name.lower()
                if base in ("knoxville.txt", "boise.txt"):
                    source_bytes[base] = case_zip.read(member)
    else:
        base = Path(filename).name.lower()
        if base in ("knoxville.txt", "boise.txt"):
            source_bytes[base] = blob

missing_files = {"knoxville.txt", "boise.txt"} - source_bytes.keys()
if missing_files:
    raise FileNotFoundError(f"Missing source file(s): {sorted(missing_files)}. Upload the original case ZIP or both TXT files, then rerun Step 2.")
print("Ready:", ", ".join(f"{k} ({len(v):,} bytes)" for k, v in sorted(source_bytes.items())))

## Step 3 — Import both markets and preserve the originals

Requirement 1: read the tab-delimited sources, add `Market`, combine rows, and record their original positions. We preserve every survey response, including possible duplicates. The **source row** is a traceability aid, not an original survey field.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

ORIGINAL_FIELDS = ["Respondent", "Age", "Regularly Consume Crispy Spuds?", "Crispiness", "Saltiness", "Sweetness", "Flavor", "Texture", "Value", "Overall"]
NUMERIC_FIELDS = ["Respondent", "Age", "Crispiness", "Saltiness", "Sweetness", "Flavor", "Texture", "Value", "Overall"]
MARKET_FILES = [("Knoxville", "knoxville.txt"), ("Boise", "boise.txt")]

market_frames = []
for market, filename in MARKET_FILES:
    incoming = pd.read_csv(io.BytesIO(source_bytes[filename]), sep="\t", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    incoming.columns = incoming.columns.str.strip()
    if list(incoming.columns) != ORIGINAL_FIELDS:
        raise ValueError(f"{filename} has unexpected columns: {list(incoming.columns)}. Expected: {ORIGINAL_FIELDS}")
    incoming = incoming.apply(lambda col: col.str.strip())
    incoming["Market"] = market
    incoming["Source Row"] = np.arange(2, len(incoming)+2)  # row number in original TXT, counting header
    market_frames.append(incoming)

raw = pd.concat(market_frames, ignore_index=True)
raw["Combined Row"] = np.arange(2, len(raw)+2)
print("Imported raw records:", len(raw))
print("Market counts:")
display(raw.groupby("Market", as_index=False).size().rename(columns={"size":"Raw records"}))
display(raw.head(5))

## Step 4 — Audit missing responses and invalid values

Requirement 2: report every record/field with a missing value. Blank survey responses are **flagged**, not automatically filled. Invalid ratings and ages are reported separately for review. Each report identifies `Market` and `Respondent`.

In [ ]:
missing_rows = []
for _, row in raw.iterrows():
    for field in ORIGINAL_FIELDS:
        if str(row[field]).strip() == "":
            missing_rows.append({"Market":row["Market"], "Respondent":row["Respondent"], "Source Row":int(row["Source Row"]), "Field":field, "Issue":"Missing response", "Proposed Action":"Investigate; do not invent a response"})
missing_report = pd.DataFrame(missing_rows, columns=["Market","Respondent","Source Row","Field","Issue","Proposed Action"])
missing_counts = pd.DataFrame({"Field":ORIGINAL_FIELDS, "Missing Values":[int(raw[c].eq("").sum()) for c in ORIGINAL_FIELDS]})

invalid_rows = []
for _, r in raw.iterrows():
    for field in NUMERIC_FIELDS:
        value = r[field]
        if value == "": continue
        numeric = pd.to_numeric(pd.Series([value]),errors="coerce").iloc[0]
        allowed = (pd.notna(numeric) and (numeric >= 0 if field == "Age" else (numeric >= 1 and numeric <= 10 if field not in ["Respondent"] else numeric > 0)))
        if field in ["Respondent", "Age"] and allowed:
            allowed = float(numeric).is_integer()
        if not allowed:
            invalid_rows.append({"Market":r["Market"],"Respondent":r["Respondent"],"Source Row":int(r["Source Row"]),"Field":field,"Observed Value":value,"Issue":"Invalid numeric response"})
    val=r["Regularly Consume Crispy Spuds?"]
    if val and val.casefold() not in ("yes","no"):
        invalid_rows.append({"Market":r["Market"],"Respondent":r["Respondent"],"Source Row":int(r["Source Row"]),"Field":"Regularly Consume Crispy Spuds?","Observed Value":val,"Issue":"Unexpected Yes/No value"})
invalid_report=pd.DataFrame(invalid_rows,columns=["Market","Respondent","Source Row","Field","Observed Value","Issue"])
print(f"Rows with at least one missing original value: {missing_report[['Market','Source Row']].drop_duplicates().shape[0] if len(missing_report) else 0}")
print(f"Missing fields (cells): {len(missing_report)} | Invalid values: {len(invalid_report)}")
display(missing_counts)
display(missing_report.head(12))
if not invalid_report.empty: display(invalid_report.head(12))

## Step 5 — Identify duplicate candidates (do not delete)

Requirement 3: list fully identical source records and any repeated `Market + Respondent` identifiers. Two different markets may both have a respondent numbered 1; that alone is **not** duplication. The `Duplicate Report` flags candidates for a human decision.

In [ ]:
# Compare original fields PLUS market; do not include technical row-location columns.
duplicate_keys = ORIGINAL_FIELDS + ["Market"]
exact_mask = raw.duplicated(subset=duplicate_keys, keep=False)
duplicate_report = raw.loc[exact_mask, ["Market","Respondent","Source Row","Combined Row"] + ORIGINAL_FIELDS[1:]].copy()
duplicate_report.insert(3, "Issue", "Identical values in all original fields within market")
duplicate_report.insert(4, "Action", "Investigate before deletion")

id_mask = raw.duplicated(subset=["Market","Respondent"],keep=False)
repeated_ids = raw.loc[id_mask,["Market","Respondent","Source Row","Combined Row"]].copy()
repeated_ids["Issue"]="Repeated Market + Respondent identifier; compare responses"
print("Exact duplicate extra rows:", int(raw.duplicated(subset=duplicate_keys).sum()))
print("Rows within exact-duplicate groups:", int(exact_mask.sum()))
print("Rows with repeated Market + Respondent:",int(id_mask.sum()))
display(duplicate_report.head(16))
print("Records are still preserved in Combined Data.")

## Step 6 — Create the Taste measure, safely

Requirement 4: `Taste = (Saltiness + Sweetness + Flavor) / 3` **only when all three scores are present and valid**. Otherwise, the corresponding Excel cell shows `REVIEW`. Excel formulas are written into the final workbook.

In [ ]:
prepared = raw.copy()
for field in NUMERIC_FIELDS:
    prepared[field] = pd.to_numeric(prepared[field].replace("", np.nan), errors="coerce")

score_fields = ["Saltiness","Sweetness","Flavor"]
complete_taste = prepared[score_fields].notna().all(axis=1) & ((prepared[score_fields] >= 1) & (prepared[score_fields] <= 10)).all(axis=1)
prepared["Taste"] = prepared[score_fields].mean(axis=1).where(complete_taste)
prepared["Taste Status"] = np.where(complete_taste,"Calculated","REVIEW: missing/invalid component")
print("Taste calculated:",int(complete_taste.sum()),"| Taste flagged:",int((~complete_taste).sum()))
display(prepared[["Market","Respondent","Saltiness","Sweetness","Flavor","Taste","Taste Status"]].head(8))

## Step 7 — Extract the target group

Requirement 5: select respondents **younger than 35** who answered **No** to regular product consumption. Both conditions must hold. This output includes `Market` and `Taste`, as requested. The source rows remain unmodified.

In [ ]:
is_young = prepared["Age"].lt(35) & prepared["Age"].ge(0)
is_nonregular = raw["Regularly Consume Crispy Spuds?"].str.casefold().eq("no")
young_no = prepared.loc[is_young & is_nonregular, ORIGINAL_FIELDS+["Market","Taste","Taste Status"]].copy()
print("Target segment records:",len(young_no))
display(young_no.head(8))

## Step 8 — Review the audit and prepare the Excel workbooks

The report contains source counts, exceptions, a change log, and an original-data sheet. **No original records are deleted.** The `Combined Data` worksheet includes actual Excel formulas in the Taste column. Students should check the results and write their own brief managerial recommendations.

In [ ]:
from datetime import datetime, timezone
import json, textwrap

output_dir = Path("/content/BIAN310_Usman_Outputs") if IN_COLAB else Path("./BIAN310_Usman_Outputs")
output_dir.mkdir(parents=True, exist_ok=True)

quality_summary = pd.DataFrame([
    ["Student",STUDENT_NAME],["Assignment",ASSIGNMENT],["Raw Knoxville records",len(market_frames[0])],
    ["Raw Boise records",len(market_frames[1])],["Combined raw records",len(raw)],
    ["Rows with missing original responses",len(missing_report[["Market","Source Row"]].drop_duplicates())],
    ["Missing original values",len(missing_report)],["Invalid values",len(invalid_report)],
    ["Extra exact duplicate rows",int(raw.duplicated(subset=duplicate_keys).sum())],
    ["Rows in exact duplicate groups",int(exact_mask.sum())],
    ["Taste review flags",int((~complete_taste).sum())],
    ["Under 35 / No segment size",len(young_no)],
    ["Source rows deleted",0]
], columns=["Metric","Value"])

change_log = pd.DataFrame([
    ["1","Imported two TSV files","Added Market and source row audit columns","All raw responses preserved"],
    ["2","Missing-value check","Flagged missing data in report","No imputation"],
    ["3","Duplicate check","Flagged exact duplicate groups and repeated IDs","No deletion"],
    ["4","Taste calculation","Added guarded 3-score mean","REVIEW on missing/invalid components"],
    ["5","Target segment","Filtered Age < 35 and regular consumption = No","Source remains intact"]
],columns=["Step","Operation","Automated Action","Decision / Rule"])

review_template = f"""BIAN 310 | Module 3 - Usman Solutions
Student: {STUDENT_NAME}
Assignment: {ASSIGNMENT}

STUDENT MANAGEMENT SUMMARY (write your own 150-200 words):
1. Describe the combined markets and raw record count.
2. Report missing-value and duplicate findings and recommended investigations.
3. Explain the guarded Taste formula and any REVIEW entries.
4. Explain the under-35 / No segment.
5. Recommend one next action before analysis begins.

REFLECTION QUESTIONS:
- Why shouldn't every repeated row be deleted automatically?
- Why should AVERAGE not ignore missing Taste components here?
- What could automation miss that a person should still verify?

ACADEMIC INTEGRITY: Identify any AI/tool help used and verify the outputs.
"""

workbook_path = output_dir/"porter_hinkle.xlsx"
segment_path = output_dir/"young_no.xlsx"
with pd.ExcelWriter(workbook_path,engine="xlsxwriter", datetime_format="yyyy-mm-dd") as writer:
    # Exact required workbook with transparent audit tabs
    combined_cols=ORIGINAL_FIELDS+["Market","Taste"]
    excel_combined=prepared[combined_cols].copy()
    excel_combined.to_excel(writer,sheet_name="Combined Data",index=False)
    raw[ORIGINAL_FIELDS+["Market","Source Row"]].to_excel(writer,sheet_name="Original Combined",index=False)
    missing_report.to_excel(writer,sheet_name="Missing Values Report",index=False)
    missing_counts.to_excel(writer,sheet_name="Missing Counts",index=False)
    duplicate_report.to_excel(writer,sheet_name="Duplicate Report",index=False)
    repeated_ids.to_excel(writer,sheet_name="Repeated IDs",index=False)
    invalid_report.to_excel(writer,sheet_name="Invalid Values",index=False)
    quality_summary.to_excel(writer,sheet_name="Data Quality Summary",index=False)
    change_log.to_excel(writer,sheet_name="Cleaning Log",index=False)
    pd.DataFrame({"Management Summary / Reflection":review_template.splitlines()}).to_excel(writer,sheet_name="Management Summary",index=False)
    book=writer.book
    header_fmt=book.add_format({"bold":True,"font_color":"#FFFFFF","bg_color":"#174A73","border":0,"text_wrap":True,"valign":"vcenter"})
    num_fmt=book.add_format({"num_format":"0.00"})
    good_fmt=book.add_format({"num_format":"0.00","bg_color":"#E7F4EE"})
    review_fmt=book.add_format({"font_color":"#B45309","bg_color":"#FFF1D6"})
    for sheet_name, sheet in writer.sheets.items():
        sheet.freeze_panes(1,0)
        sheet.set_default_row(18)
        sheet.set_row(0,32)
        sheet.set_column(0,0,18)
        sheet.set_column(1,1,15)
        sheet.set_column(2,13,22)
        if sheet_name in ("Cleaning Log","Management Summary"):
            sheet.set_column(1,4,48)
        sheet.autofilter(0,0,0, 0) if False else None
        sheet.set_tab_color("#2B86B0" if sheet_name=="Combined Data" else "#84B9CE")
        # Apply header format using header text already written
        data_frame={"Combined Data":excel_combined,"Original Combined":raw[ORIGINAL_FIELDS+["Market","Source Row"]],"Missing Values Report":missing_report,
                    "Missing Counts":missing_counts,"Duplicate Report":duplicate_report,"Repeated IDs":repeated_ids,"Invalid Values":invalid_report,
                    "Data Quality Summary":quality_summary,"Cleaning Log":change_log,
                    "Management Summary":pd.DataFrame({"Management Summary / Reflection":review_template.splitlines()})}[sheet_name]
        for j,colname in enumerate(data_frame.columns):sheet.write(0,j,colname,header_fmt)
        if len(data_frame):sheet.autofilter(0,0,len(data_frame),len(data_frame.columns)-1)
    sheet=writer.sheets["Combined Data"]
    # Original column positions: E=Saltiness, F=Sweetness, G=Flavor; L=Taste
    for i, numeric in enumerate(prepared["Taste"],start=2):
        formula=f'=IF(AND(COUNT(E{i}:G{i})=3,MIN(E{i}:G{i})>=1,MAX(E{i}:G{i})<=10),AVERAGE(E{i}:G{i}),"REVIEW")'
        value=float(numeric) if pd.notna(numeric) else "REVIEW"
        sheet.write_formula(i-1,11,formula,good_fmt if pd.notna(numeric) else review_fmt,value)

with pd.ExcelWriter(segment_path, engine="xlsxwriter") as writer:
    young_no.to_excel(writer,sheet_name="Young Nonregular",index=False)
    sheet=writer.sheets["Young Nonregular"]
    sheet.freeze_panes(1,0)
    sheet.set_column(0,12,20)
    sheet.set_row(0,32)
    for c,name in enumerate(young_no.columns):sheet.write(0,c,name,header_fmt)
    if len(young_no):sheet.autofilter(0,0,len(young_no),len(young_no.columns)-1)

missing_report.to_csv(output_dir/"missing_values_report.csv",index=False)
duplicate_report.to_csv(output_dir/"duplicate_report.csv",index=False)
quality_summary.to_csv(output_dir/"retail_data_quality_summary.csv",index=False)
(output_dir/"student_management_summary_template.txt").write_text(review_template,encoding="utf-8")
for key,blob in source_bytes.items():
    (output_dir/key).write_bytes(blob)

print("Excel files created:",workbook_path.name,segment_path.name)
display(quality_summary)

## Step 9 — Dashboard: survey findings and data quality problems

**This dashboard is generated automatically from the uploaded Knoxville and Boise source files.** It appears below in Colab and is also saved as a PNG image and a standalone HTML report in the final ZIP. It highlights:

- survey volume by market;
- missing and invalid responses (by field);
- exact-duplicate *candidates* and review flags;
- calculated versus incomplete Taste scores;
- respondents younger than 35 who selected No;
- average valid Taste score by market.

**Interpretation warning:** The visualizations are descriptive, not causal. Duplicate candidates are **not** automatically removed, and missing ratings are not fabricated. A lower sample average does not by itself explain the cause of satisfaction or imply that any records should be deleted.


In [ ]:
# Dashboard: built from the reports and frames above; no additional uploads needed.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import base64
from IPython.display import display, HTML

market_counts = raw.groupby("Market").size().reindex(["Knoxville", "Boise"], fill_value=0)
missing_fields = missing_counts.set_index("Field")["Missing Values"]
missing_fields = missing_fields[missing_fields > 0].sort_values(ascending=False)
invalid_count = int(len(invalid_report))
extra_exact_rows = int(raw.duplicated(subset=duplicate_keys).sum())
repeat_id_groups = int(raw.loc[id_mask, ["Market", "Respondent"]].drop_duplicates().shape[0])
flagged_taste = int((~complete_taste).sum())
valid_taste = int(complete_taste.sum())
segment_market = young_no.groupby("Market").size().reindex(market_counts.index, fill_value=0)
taste_market = prepared.groupby("Market")["Taste"].mean().reindex(market_counts.index)

# Dashboard KPIs are counted directly from the source records; they are not adjusted by deleting candidates.
kpi_data = {
    "Raw survey records": int(len(raw)),
    "Markets": int(raw["Market"].nunique()),
    "Missing response cells": int(len(missing_report)),
    "Invalid response cells": invalid_count,
    "Extra exact-duplicate rows": extra_exact_rows,
    "Taste review flags": flagged_taste,
    "Under 35 + No responses": int(len(young_no)),
}
dashboard_kpis = pd.DataFrame(kpi_data.items(), columns=["Metric", "Value"])
dashboard_kpis.to_csv(output_dir / "dashboard_kpis.csv", index=False)

plt.rcParams.update({"font.size": 10})
fig = plt.figure(figsize=(15, 11), facecolor="white")
layout = fig.add_gridspec(4, 2, height_ratios=[0.8, 2.6, 2.6, 2.6], hspace=0.52, wspace=0.28)
ax_header = fig.add_subplot(layout[0, :]); ax_header.axis("off")
ax_header.text(0.0, 0.93, "USMAN SOLUTIONS  |  DATA QUALITY & SURVEY DASHBOARD", fontsize=19, fontweight="bold", color="#153451", va="top")
ax_header.text(0.0, 0.36, f"BIAN 310 • Module 3   |   Prepared by {STUDENT_NAME}   |   {ASSIGNMENT}", fontsize=10.5, color="#526579", va="top")

ax1 = fig.add_subplot(layout[1,0]); ax2 = fig.add_subplot(layout[1,1])
ax3 = fig.add_subplot(layout[2,0]); ax4 = fig.add_subplot(layout[2,1])
ax5 = fig.add_subplot(layout[3,0]); ax6 = fig.add_subplot(layout[3,1])

def style(ax, title, xlabel=None):
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold", pad=12, color="#153451")
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", color="#DFE7ED", linewidth=0.65, zorder=0)
    ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel)

def annotate_bars(ax, bars, fmt="{:.0f}"):
    for b in bars:
        h=b.get_height()
        ax.annotate(fmt.format(h), (b.get_x()+b.get_width()/2, h), xytext=(0,4), textcoords="offset points", ha="center", va="bottom", fontsize=9)

bars=ax1.bar(market_counts.index, market_counts.values, color=["#277EAA", "#55A9A1"], width=0.52)
style(ax1, "1 | Raw survey records by market")
ax1.set_ylim(0, max(market_counts.values)*1.18); annotate_bars(ax1,bars)

if len(missing_fields):
    labels=list(missing_fields.index)
    bars=ax2.barh(labels[::-1], missing_fields.values[::-1], color="#E7A44B")
    for b in bars: ax2.text(b.get_width()+0.04, b.get_y()+b.get_height()/2, f"{int(b.get_width())}", va="center")
    ax2.set_xlim(0,max(missing_fields.values)*1.3+0.1)
else:
    ax2.text(0.5, 0.5, "No blank responses detected", ha="center", va="center", transform=ax2.transAxes)
style(ax2,"2 | Missing responses by field")
ax2.grid(axis="x", color="#DFE7ED", linewidth=0.65); ax2.grid(axis="y", visible=False)

issues=["Missing cells","Invalid cells","Extra duplicate rows","Taste review flags"]
counts=[len(missing_report),invalid_count,extra_exact_rows,flagged_taste]
bars=ax3.bar(issues,counts,color=["#DD9650","#C96363","#C97C84","#8877C3"],width=0.56)
style(ax3,"3 | Data-quality issue counts (different measures)")
ax3.tick_params(axis="x", labelrotation=20)
ax3.set_ylim(0,max(1,max(counts))*1.28); annotate_bars(ax3,bars)

bars=ax4.bar(["Calculated", "Review required"],[valid_taste,flagged_taste],color=["#55A9A1","#D58B4B"],width=0.55)
style(ax4,"4 | Taste score readiness")
ax4.set_ylim(0,max(1,valid_taste)*1.15); annotate_bars(ax4,bars)

bars=ax5.bar(segment_market.index,segment_market.values,color=["#277EAA", "#55A9A1"],width=0.52)
style(ax5,"5 | Under-35 / No segment by market")
ax5.set_ylim(0,max(1,max(segment_market.values))*1.25); annotate_bars(ax5,bars)

valid_means=taste_market.dropna()
if len(valid_means):
    bars=ax6.bar(valid_means.index,valid_means.values,color=["#277EAA","#55A9A1"][:len(valid_means)],width=0.52)
    annotate_bars(ax6,bars,"{:.2f}")
else:
    ax6.text(0.5,0.5,"No complete Taste scores",ha="center",va="center",transform=ax6.transAxes)
style(ax6,"6 | Mean valid Taste rating by market")
ax6.set_ylim(0,10.5)
fig.text(0.04,0.005,"Source: provided Knoxville and Boise surveys. Suspected duplicates retained; Taste averages exclude incomplete/invalid components.",fontsize=9,color="#526579")

png_path=output_dir/"usman_data_quality_dashboard.png"
fig.savefig(png_path,dpi=170,bbox_inches="tight",facecolor="white")
plt.show()

# Write a readable HTML dashboard with the same chart and an accessible metric table.
encoded=base64.b64encode(png_path.read_bytes()).decode("ascii")
rows="".join(f"<tr><td>{metric}</td><td>{value:,}</td></tr>" for metric,value in kpi_data.items())
html_report=f"""<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width, initial-scale=1'>
<title>Usman Solutions — Data Quality Dashboard</title>
<style>body{{font-family:Arial,sans-serif;margin:0 auto;padding:28px;max-width:1250px;color:#153451;background:#f5f7fa}}
main{{background:white;border-radius:16px;padding:24px;box-shadow:0 3px 18px #18375116}}
h1{{margin:0 0 6px}}p{{color:#526579;line-height:1.5}}img{{width:100%;height:auto}}
table{{border-collapse:collapse;width:100%;max-width:690px}}td,th{{padding:9px 12px;border-bottom:1px solid #dae3eb;text-align:left}}td:last-child{{text-align:right}}
.notice{{border-left:4px solid #e0a34d;padding:10px 14px;background:#fff8eb}}</style></head><body><main>
<h1>Usman Solutions | Data Quality Dashboard</h1><p>Student: {STUDENT_NAME} | Assignment: {ASSIGNMENT}</p>
<img src='data:image/png;base64,{encoded}' alt='Six-panel survey and data quality dashboard'>
<h2>Dashboard measures</h2><table><thead><tr><th>Metric</th><th>Value</th></tr></thead><tbody>{rows}</tbody></table>
<p class='notice'>Counts use preserved original source rows. Duplicate candidates are flagged, not removed. Incomplete or invalid Taste components are not averaged. Review audit files before drawing conclusions.</p>
</main></body></html>"""
html_path=output_dir/"usman_data_quality_dashboard.html"
html_path.write_text(html_report, encoding="utf-8")

# Management interpretation is a guided template, not an automatically asserted conclusion.
findings=(f"DATA QUALITY DASHBOARD — AUTOMATED OBSERVATIONS\n"
          f"Student: {STUDENT_NAME}\nAssignment: {ASSIGNMENT}\n\n"
          f"Raw records: {len(raw)} (Knoxville: {market_counts.get('Knoxville',0)}, Boise: {market_counts.get('Boise',0)})\n"
          f"Missing response cells: {len(missing_report)}; invalid response cells: {invalid_count}\n"
          f"Extra exact-duplicate rows: {extra_exact_rows}; repeated market/respondent groups: {repeat_id_groups}\n"
          f"Taste score review flags: {flagged_taste}; complete Taste scores: {valid_taste}\n"
          f"Under-35 and nonregular group: {len(young_no)}\n\n"
          "REQUIRED HUMAN REVIEW:\n"
          "1. Which missing values affect a downstream analysis and why?\n"
          "2. Are the duplicate candidates genuine errors? Cite evidence before deciding.\n"
          "3. What do valid Taste scores suggest, and what can't they establish?\n"
          "4. What recommendation would you give the Analytics Division?\n")
(output_dir/"dashboard_findings_and_review.txt").write_text(findings,encoding="utf-8")

print("DASHBOARD FILES GENERATED:")
print("  -",png_path.name)
print("  -",html_path.name)
print("  - dashboard_kpis.csv")
print("  - dashboard_findings_and_review.txt")
display(dashboard_kpis)
display(HTML("<p><strong>Dashboard created.</strong> Open the PNG/HTML files from the downloaded ZIP to present the findings.</p>"))


## Step 10 — Package everything and automatically download the ZIP

The ZIP includes the required `porter_hinkle.xlsx` and `young_no.xlsx`, audit CSVs, source `.txt` files, and a management-summary template. If your browser blocks the automatic download, allow downloads for Colab and rerun only this cell.

In [ ]:
import zipfile
zip_path = output_dir.parent/f"BIAN310_Module3_Usman_{SAFE_NAME}_Submission_Package.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as outzip:
    for item in sorted(output_dir.iterdir()):
        if item.is_file(): outzip.write(item,arcname=item.name)
with zipfile.ZipFile(zip_path) as checkzip:
    bad=checkzip.testzip()
    if bad: raise RuntimeError(f"ZIP integrity check failed: {bad}")
    print(f"Created {zip_path.name} ({zip_path.stat().st_size:,} bytes)")
    print("Files:",*checkzip.namelist(),sep="\n  - ")
if IN_COLAB:
    print("Starting automatic download…")
    files.download(str(zip_path))
else:
    print("Local execution: find ZIP at",zip_path.resolve())

## Before submitting (student checklist)

- [ ] I checked the `Combined Data` sheet, source-market counts, and Taste formulas.
- [ ] I read the `Missing Values Report` and `Duplicate Report` and understand that exceptions require investigation.
- [ ] I verified `young_no.xlsx` contains only respondents under 35 who answered **No**.
- [ ] I wrote my **own** manager summary using the template and saved it in the workbook or as an accompanying document.
- [ ] I submitted the files required by my instructor, and disclosed any AI/automation assistance.

**Why not automatically delete duplicates?** Two identical survey responses could reflect an import error or separate valid records. Retain originals until a manager verifies the policy. **Why flag Taste?** Missing ratings must not silently change a three-component average into a two-component average.

**Submission reminder:** Open the workbooks after downloading the ZIP, review the exception reports, complete your management summary in your own words, and follow your instructor’s LMS submission directions.
